# Lab 1: Creating Local MCP Servers

In this lab, you'll build an MCP server from scratch using Python and connect it to Claude Code and VS Code.

## What is an MCP Server?

An MCP (Model Context Protocol) server exposes **tools**, **resources**, and **prompts** to AI assistants. Think of it as a plugin system — your AI assistant discovers what tools are available and can call them when needed.

```
┌──────────────┐     stdio/HTTP      ┌──────────────┐
│  AI Client   │ ◄──────────────────► │  MCP Server  │
│ (Claude Code │     JSON-RPC 2.0     │  (Your Code) │
│  / VS Code)  │                      │              │
└──────────────┘                      └──────┬───────┘
                                             │
                                      ┌──────┴───────┐
                                      │  Local Tools  │
                                      │  SSH, Files,  │
                                      │  Ports, etc.  │
                                      └──────────────┘
```

## MCP Primitives

| Primitive | What it does | Example |
|-----------|-------------|----------|
| **Tool** | Executable function the AI can call | `run_remote_command(host, command)` |
| **Resource** | Read-only data the AI can access | `file://config.yaml` |
| **Prompt** | Pre-built instruction templates | `code_review` checklist |

---
## Part 1: Hello World MCP Server

Let's start with the absolute minimum MCP server — one tool, five lines of code.

In [ ]:
# First, verify FastMCP is installed
import fastmcp
print(f"FastMCP version: {fastmcp.__version__}")

In [ ]:
# Minimal MCP server — this is all you need!
from fastmcp import FastMCP

mcp = FastMCP("hello-world")

@mcp.tool()
def greet(name: str) -> str:
    """Say hello to someone."""
    return f"Hello, {name}! Welcome to MCP."

# In a script, you'd run: mcp.run(transport="stdio")
# But in a notebook, let's just verify the tool is registered:
print("Registered tools:", [t.name for t in mcp._tool_manager.tools.values()])

That's it. The `@mcp.tool()` decorator:
1. Registers the function as an MCP tool
2. Uses the **type hints** to generate the JSON schema automatically
3. Uses the **docstring** as the tool description (what the AI reads to decide when to use it)

> **Key insight**: The better your docstring and type hints, the better the AI will use your tool.

---
## Part 2: Building Real Tools

Now let's look at the actual dev tools we'll use. Open `servers/dev_tools_stdio.py` and walk through each tool.

### Tool 1: `run_remote_command`
Executes a shell command on a remote machine via SSH.

In [ ]:
import subprocess

def run_remote_command(host: str, command: str, user: str = "") -> str:
    """Run a shell command on a remote machine via SSH."""
    ssh_target = f"{user}@{host}" if user else host
    try:
        result = subprocess.run(
            ["ssh", "-o", "StrictHostKeyChecking=accept-new", "-o", "ConnectTimeout=10",
             ssh_target, command],
            capture_output=True, text=True, timeout=30,
        )
        output = result.stdout.strip()
        errors = result.stderr.strip()
        parts = []
        if output:
            parts.append(f"STDOUT:\n{output}")
        if errors:
            parts.append(f"STDERR:\n{errors}")
        if result.returncode != 0:
            parts.append(f"Exit code: {result.returncode}")
        return "\n\n".join(parts) if parts else "(no output)"
    except subprocess.TimeoutExpired:
        return f"ERROR: Command timed out after 30 seconds on {host}"
    except FileNotFoundError:
        return "ERROR: ssh not found on PATH"

# Test locally (this will fail if you don't have SSH access — that's OK!)
# Uncomment and modify with your host:
# print(run_remote_command("your-server", "uptime", "your-user"))

### Tool 2: `file_tree`
Generates a clean directory tree. Useful for quickly understanding project structure.

In [ ]:
from pathlib import Path

def file_tree(directory: str = ".", max_depth: int = 3,
              ignore_patterns: str = ".git,node_modules,__pycache__,.venv") -> str:
    """Generate a clean directory tree listing."""
    root = Path(directory).resolve()
    if not root.is_dir():
        return f"ERROR: {directory} is not a valid directory."
    ignores = set(p.strip() for p in ignore_patterns.split(","))
    lines = [f"{root.name}/"]

    def _walk(path, prefix, depth):
        if depth > max_depth:
            return
        try:
            entries = sorted(path.iterdir(), key=lambda e: (not e.is_dir(), e.name.lower()))
        except PermissionError:
            return
        dirs = [e for e in entries if e.is_dir() and e.name not in ignores]
        files = [e for e in entries if e.is_file() and e.name not in ignores]
        items = dirs + files
        for i, item in enumerate(items):
            is_last = i == len(items) - 1
            connector = "└── " if is_last else "├── "
            lines.append(f"{prefix}{connector}{item.name}{'/' if item.is_dir() else ''}")
            if item.is_dir():
                extension = "    " if is_last else "│   "
                _walk(item, prefix + extension, depth + 1)

    _walk(root, "", 1)
    return "\n".join(lines)

# Test it on this repo
print(file_tree("../..", max_depth=2))

### Tool 3: `port_checker`
Checks which common dev ports are in use — great for debugging "address already in use" errors.

In [ ]:
import socket

def port_checker(ports: str = "3000,5000,8000,8080,9000") -> str:
    """Check which common development ports are in use."""
    results = []
    for port_str in ports.split(","):
        port = int(port_str.strip())
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(0.5)
            in_use = s.connect_ex(("127.0.0.1", port)) == 0
        status = "IN USE" if in_use else "FREE"
        results.append(f"  :{port}  {status}")
    return "Port Status:\n" + "\n".join(results)

print(port_checker())

---
## Part 3: Adding Prompts (Review Context)

MCP **prompts** are pre-built instruction templates. They give the AI rich context for specific tasks like code reviews.

```python
@mcp.prompt()
def code_review() -> str:
    """Code review checklist."""
    return """You are performing a code review. Evaluate against:
    ## Correctness
    - Does the code do what it's supposed to do?
    ..."""
```

Our server includes 4 review prompts:
- `code_review` — General code quality
- `security_review` — OWASP-focused security check
- `design_review` — Architecture evaluation
- `networking_review` — Network/firewall configuration

In Claude Code, you can use these by asking: *"Use the security_review prompt to review this file."*

---
## Part 4: Testing with FastMCP Inspector

FastMCP includes a built-in web inspector for testing your server without connecting to an AI client.

```bash
# From the repo root:
fastmcp dev labs/lab1-local-mcp-servers/servers/dev_tools_stdio.py
```

This opens a web UI where you can:
- See all registered tools and prompts
- Call tools with custom arguments
- View the JSON-RPC messages being exchanged

> **Try it now!** Open a terminal and run the command above.

---
## Part 5: Connecting to Claude Code

### Step 1: Create the config file

Claude Code reads `.mcp.json` from the project root. Copy our template:

```bash
cp labs/lab1-local-mcp-servers/configs/mcp.json .mcp.json
```

Or create it manually:

```json
{
  "mcpServers": {
    "dev-tools": {
      "command": "python",
      "args": ["labs/lab1-local-mcp-servers/servers/dev_tools_stdio.py"]
    }
  }
}
```

### Step 2: Start Claude Code

```bash
cd /path/to/local_mcp
claude
```

### Step 3: Verify tools are loaded

Ask Claude: *"What MCP tools do you have available?"*

You should see `file_tree`, `port_checker`, and `run_remote_command`.

### Step 4: Try them out!

- *"Show me the file tree of this project"*
- *"Check which ports are in use on my machine"*
- *"Use the code_review prompt to review servers/dev_tools_stdio.py"*

---
## Part 6: Connecting to VS Code

### Step 1: Create the config file

VS Code reads `.vscode/mcp.json` from the workspace root:

```bash
mkdir -p .vscode
cp labs/lab1-local-mcp-servers/configs/vscode_mcp.json .vscode/mcp.json
```

### Step 2: Restart VS Code

After creating/editing the config, restart VS Code for it to pick up the MCP server.

### Step 3: Use in Copilot Chat

Open Copilot Chat (Ctrl+Shift+I) and look for the tools icon. Your MCP tools should be listed there.

> **Note**: VS Code MCP support requires version 1.99+. Check with `code --version`.

---
## Part 7: HTTP Transport

So far we used **STDIO** — the server runs as a child process and communicates via stdin/stdout. This is great for local use.

**HTTP transport** runs the server on a network port, making it accessible from other machines.

```
STDIO:  Client ──stdin/stdout──► Server (same machine, one client)
HTTP:   Client ──HTTP/HTTPS────► Server (any machine, multiple clients)
```

### Running the HTTP server

```bash
python labs/lab1-local-mcp-servers/servers/dev_tools_http.py
# → Server starts at http://0.0.0.0:9000
```

### How it works

Look at `dev_tools_http.py` — it's just 3 lines! It imports the same `mcp` instance and changes the transport:

```python
from dev_tools_stdio import mcp

if __name__ == "__main__":
    mcp.run(transport="streamable-http", host="0.0.0.0", port=9000)
```

The tool logic is 100% identical — only the transport changes. This is the power of MCP's transport abstraction.

---
## Exercise: Add Your Own Tool

Try adding a custom tool to `dev_tools_stdio.py`. Ideas:

- `disk_usage(path)` — Check disk space for a directory
- `git_status(repo_path)` — Show git status summary
- `env_checker(vars)` — Check if environment variables are set
- `http_ping(url)` — Check if a URL is reachable

Template:

```python
@mcp.tool()
def my_tool(param: str) -> str:
    """Describe what this tool does — the AI reads this!"""
    # Your logic here
    return "result"
```

After adding it, restart the MCP server (or re-run `fastmcp dev`) to see your new tool.

---
## Summary

| Concept | What You Learned |
|---------|------------------|
| FastMCP | Python SDK for building MCP servers with decorators |
| `@mcp.tool()` | Register a function as a callable tool |
| `@mcp.prompt()` | Register pre-built instruction templates |
| STDIO transport | Local, single-client, via stdin/stdout |
| HTTP transport | Network-accessible, multi-client, via HTTP |
| `.mcp.json` | Claude Code MCP configuration |
| `.vscode/mcp.json` | VS Code MCP configuration |

**Next**: [Lab 2 — External MCP Sources](../lab2-external-mcp-sources/README.md)